In [ ]:
import os
import sys
import torch

RUN = 'consensus'
CODE = '/kaggle/tmp/knee-mri-findings'
if RUN not in ('report', 'consensus', 'consensus_full'):
    raise RuntimeError(f'RUN must be report, consensus or consensus_full, not {RUN}')
if torch.cuda.device_count() == 0:
    raise RuntimeError('no GPU: choose the accelerator GPU T4 x2')
target = 'report' if RUN == 'report' else 'consensus'
crops = labels = pretrained = None
for root, dirs, files in os.walk('/kaggle/input'):
    dirs[:] = sorted(d for d in dirs if d not in {'train_series', 'test_series', 'crops', 'models', 'segmentation', 'wheels'})
    if crops is None and os.path.isdir(f'{root}/crops') and any(f.endswith('.npz') for f in os.listdir(f'{root}/crops')):
        crops = f'{root}/crops'
    if labels is None and os.path.isfile(f'{root}/labels/{target}.csv'):
        labels = f'{root}/labels/{target}.csv'
    if pretrained is None and 'resnet18d.pth' in files:
        pretrained = f'{root}/resnet18d.pth'
missing = [n for n, v in (('crops', crops), (f'labels/{target}.csv', labels), ('resnet18d.pth', pretrained)) if v is None]
if missing:
    raise RuntimeError(f'not found in the inputs: {missing}')
os.makedirs(f'{CODE}/knee', exist_ok=True)
os.makedirs(f'{CODE}/configs', exist_ok=True)
os.chdir(CODE)
print(f'{RUN} | GPUs {torch.cuda.device_count()} | crops {crops} | labels {labels}')

In [ ]:
%%writefile knee/__init__.py

In [ ]:
%%writefile knee/findings.py
import numpy as np

LABELS = ['ACL', 'MCL', 'Medial Meniscus', 'Lateral Meniscus', 'Medial OA', 'Lateral OA', 'PF OA', 'Effusion', 'Synovitis',
          "Baker's", 'Contusion', 'Fracture']

FOCUS = {
    'ACL': (['notch'], ['lateral_TF', 'medial_TF', 'joint']),
    'MCL': (['MCL'], ['medial_TF', 'bones']),
    'Medial Meniscus': (['medial_TF'], ['notch', 'MCL']),
    'Lateral Meniscus': (['lateral_TF'], ['notch']),
    'Medial OA': (['medial_TF'], ['bones', 'notch', 'MCL']),
    'Lateral OA': (['lateral_TF'], ['bones', 'notch']),
    'PF OA': (['PF'], ['joint']),
    'Effusion': (['joint', 'PF'], ['popliteal', 'notch']),
    'Synovitis': (['joint', 'PF'], ['notch', 'popliteal']),
    "Baker's": (['popliteal'], ['joint', 'medial_TF']),
    'Contusion': (['bones', 'medial_TF', 'lateral_TF'], ['PF', 'notch']),
    'Fracture': (['bones', 'medial_TF', 'lateral_TF', 'PF'], ['notch', 'MCL']),
}


def relevance(crop_names):
    R = np.full((len(LABELS), len(crop_names)), 2, np.int64)
    for i, label in enumerate(LABELS):
        main, near = FOCUS[label]
        for j, name in enumerate(crop_names):
            region = name.split('__')[0]
            R[i, j] = 0 if region in main else 1 if region in near else 2
    return R


def multilabel_folds(Y, k=5, seed=42):
    rng = np.random.default_rng(seed)
    Y = np.asarray(Y, bool)
    n, L = Y.shape
    fold = np.full(n, -1)
    need_l = np.tile(Y.sum(0) / k, (k, 1)).astype(float)
    need_f = np.full(k, n / k)
    left = np.ones(n, bool)
    while left.any():
        cnt = Y[left].sum(0)
        if cnt.max() == 0:
            for i in rng.permutation(np.where(left)[0]):
                f = int(np.argmax(need_f + rng.random(k) * 1e-06))
                fold[i] = f
                need_f[f] -= 1
                left[i] = False
            break
        col = int(np.argmin(np.where(cnt > 0, cnt, np.inf)))
        for i in rng.permutation(np.where(left & Y[:, col])[0]):
            f = int(np.lexsort((rng.random(k), need_f, need_l[:, col]))[-1])
            fold[i] = f
            need_l[f] -= Y[i]
            need_f[f] -= 1
            left[i] = False
    return fold

In [ ]:
%%writefile knee/crops.py
import os
import io
import json
import zlib
import warnings
from types import SimpleNamespace
from collections import Counter
import numpy as np
import pandas as pd
import pydicom
import cv2
from scipy import ndimage as ndi
from PIL import Image, features

VERSION = 2
OUT_PX, MAX_SLICES, MIN_SLICES = 128, 24, 3
MAX_LARGE = 16
LARGE_ROIS = ('joint', 'bones', 'popliteal', 'MCL')
MARGIN_MM, SLAB_MM, THIN_MM = 4.0, 3.0, 2.0
MIN_SERIES_IMAGES = 8
ROIS = ['medial_TF', 'lateral_TF', 'notch', 'PF', 'MCL', 'joint', 'popliteal', 'bones']
SLOTS = ['sag_nf', 'sag_fs', 'cor_fs', 'cor_nf', 'ax_fs']
VIEW = {'sag_nf': 'sag', 'sag_fs': 'sag', 'cor_fs': 'cor', 'cor_nf': 'cor', 'ax_fs': 'ax'}
CROPS = [('medial_TF', 'sag_nf'), ('medial_TF', 'sag_fs'), ('medial_TF', 'cor_fs'),
         ('lateral_TF', 'sag_nf'), ('lateral_TF', 'sag_fs'), ('lateral_TF', 'cor_fs'),
         ('notch', 'sag_nf'), ('notch', 'sag_fs'), ('notch', 'cor_fs'),
         ('MCL', 'cor_fs'), ('MCL', 'ax_fs'),
         ('PF', 'ax_fs'), ('PF', 'sag_fs'), ('PF', 'sag_nf'),
         ('popliteal', 'ax_fs'), ('popliteal', 'sag_fs'),
         ('joint', 'sag_fs'), ('joint', 'ax_fs'),
         ('bones', 'cor_fs'), ('bones', 'sag_fs'), ('bones', 'cor_nf')]
CROP_NAMES = [f'{r}__{s}' for r, s in CROPS]
ANCHOR = {'sag_nf': 'medial_TF', 'sag_fs': 'bones', 'cor_fs': 'bones', 'cor_nf': 'bones', 'ax_fs': 'joint'}
GRE_WORDS = ('FFE', 'GRE', 'SPGR', 'FLASH', 'MPRAGE', 'DESS', 'FIESTA', 'VIBE', 'WATS', 'MEDIC', 'TRUFI', 'MERGE',
             'COSMIC', 'T2*', 'T2_STAR', 'HEMO')
DERIVED_WORDS = ('MPR', 'SECONDARY', 'PROJECTION', 'REFORMAT', 'MIP', 'SUBTRACT')
WEBP = bool(features.check('webp'))


def _col(df, name, default=''):
    return df[name] if name in df.columns else pd.Series([default] * len(df), index=df.index)


def rank_series(S):
    plane = _col(S, 'Anatomical_Plane').fillna('').astype(str).str.strip().str[:3].str.lower()
    fs = pd.to_numeric(_col(S, 'Fat_Suppression', 0), errors='coerce').fillna(0).astype(int)
    txt = (_col(S, 'image_type').fillna('').astype(str) + ' ' + _col(S, 'series_desc').fillna('').astype(str)).str.upper()
    seq = _col(S, 'scanning_sequence').fillna('').astype(str).str.upper()
    tr = pd.to_numeric(_col(S, 'TR', np.nan), errors='coerce')
    te = pd.to_numeric(_col(S, 'TE', np.nan), errors='coerce')
    n = pd.to_numeric(_col(S, 'n_slices', 0), errors='coerce').fillna(0)
    acq = _col(S, 'mr_acq').fillna('').astype(str).str.upper()
    err = _col(S, 'error').fillna('').astype(str).str.strip().str.len() > 0
    gre = (seq.str.contains('GR') & ~seq.str.contains('SE')) | txt.apply(lambda t: any(w in t for w in GRE_WORDS))
    der = txt.apply(lambda t: any(w in t for w in DERIVED_WORDS))
    is3d = (acq == '3D') | (n >= 60)
    t1 = (tr < 1000).values
    pdw = ((tr >= 1000) & (te <= 60)).values
    unk = tr.isna().values
    stir = seq.str.contains('IR').values
    out = {}
    for slot in SLOTS:
        want_plane = {'sag': 'sag', 'cor': 'cor', 'ax': 'axi'}[VIEW[slot]]
        m = ((plane == want_plane) & (fs == int(slot.endswith('_fs')))).values
        if not m.any():
            continue
        if slot == 'sag_nf':
            pref = np.where(pdw, 0, np.where(t1 | unk, 1, 2))
        elif slot == 'cor_nf':
            pref = np.where(t1, 0, np.where(pdw | unk, 1, 2))
        else:
            pref = np.where(stir, 1, 0)
        K = pd.DataFrame({'uid': S.SeriesInstanceUID.astype(str).values, 'err': err.values, 'gre': gre.values,
                          'der': der.values, 'is3d': is3d.values, 'pref': pref, 'few': (n < 10).values,
                          'neg_n': -n.values})[m]
        out[slot] = K.sort_values(['err', 'gre', 'der', 'is3d', 'pref', 'few', 'neg_n'], kind='stable').uid.tolist()
    return out


def _f(v, default=None):
    try:
        return float(v)
    except Exception:
        return default


def _seq0(ds, name):
    try:
        v = ds.get(name)
        return v[0] if v is not None and len(v) else None
    except Exception:
        return None


def _decode(ds, path):
    try:
        return ds.pixel_array, False
    except Exception:
        import SimpleITK as sitk
        return sitk.GetArrayFromImage(sitk.ReadImage(path)), True


def _frame_geom(ds, k, nf):
    if nf == 1 and 'ImagePositionPatient' in ds:
        ipp, iop, ps = ds.get('ImagePositionPatient'), ds.get('ImageOrientationPatient'), ds.get('PixelSpacing')
        slope, inter = _f(ds.get('RescaleSlope', 1), 1.0), _f(ds.get('RescaleIntercept', 0), 0.0)
    else:
        per, sh = ds.get('PerFrameFunctionalGroupsSequence'), _seq0(ds, 'SharedFunctionalGroupsSequence')
        g = per[k] if per is not None and k < len(per) else None

        def grp(name):
            for src in (g, sh):
                v = _seq0(src, name) if src is not None else None
                if v is not None:
                    return v
            return None
        pp, po = grp('PlanePositionSequence'), grp('PlaneOrientationSequence')
        pm, pv = grp('PixelMeasuresSequence'), grp('PixelValueTransformationSequence')
        ipp = pp.get('ImagePositionPatient') if pp is not None else None
        iop = po.get('ImageOrientationPatient') if po is not None else ds.get('ImageOrientationPatient')
        ps = pm.get('PixelSpacing') if pm is not None else ds.get('PixelSpacing')
        src = pv if pv is not None else ds
        slope, inter = _f(src.get('RescaleSlope', 1), 1.0), _f(src.get('RescaleIntercept', 0), 0.0)
    if ipp is None or iop is None or ps is None:
        return None
    try:
        ipp, iop, ps = (np.array([float(v) for v in a]) for a in (ipp, iop, ps))
    except Exception:
        return None
    if ipp.shape != (3,) or iop.shape != (6,) or ps.shape != (2,) or ps.min() <= 0:
        return None
    return ipp, iop, ps, (slope or 1.0), (inter or 0.0)


def load_series(d):
    files = sorted(f for f in os.listdir(d) if f.lower().endswith('.dcm'))
    if not files:
        raise FileNotFoundError(f'no .dcm files in {d}')
    frames, tsx = [], Counter()
    info = dict(files=len(files), fallback=0, multiframe=0, skipped=0, mono1=0)
    for f in files:
        p = os.path.join(d, f)
        try:
            ds = pydicom.dcmread(p)
        except Exception:
            info['skipped'] += 1
            continue
        try:
            tsx[str(ds.file_meta.TransferSyntaxUID.name)] += 1
        except Exception:
            pass
        try:
            a, rescaled = _decode(ds, p)
        except Exception:
            info['skipped'] += 1
            continue
        a = np.asarray(a)
        if a.ndim == 2:
            a = a[None]
        if a.ndim != 3:
            info['skipped'] += 1
            continue
        info['fallback'] += int(rescaled)
        info['multiframe'] += int(a.shape[0] > 1)
        mono1 = str(ds.get('PhotometricInterpretation', '')).upper() == 'MONOCHROME1'
        echo, inst = _f(ds.get('EchoTime', None), 0.0) or 0.0, _f(ds.get('InstanceNumber', None), 0.0) or 0.0
        thk = _f(ds.get('SliceThickness', None), None)
        for k in range(a.shape[0]):
            g = _frame_geom(ds, k, a.shape[0])
            if g is None:
                info['skipped'] += 1
                continue
            ipp, iop, ps, slope, inter = g
            if rescaled:
                slope, inter = 1.0, 0.0
            frames.append(dict(px=a[k], ipp=ipp, iop=iop, ps=ps, slope=slope, inter=inter, mono1=mono1, echo=echo,
                               inst=inst * 10000 + k, thk=thk))
    if not frames:
        raise ValueError('no readable image in the series')

    def key(x):
        return x['px'].shape, tuple(np.round(x['iop'], 2))
    best, count = Counter(key(x) for x in frames).most_common(1)[0]
    if count < MIN_SERIES_IMAGES:
        raise ValueError(f'only {count} images with one geometry (localizer?)')
    frames = [x for x in frames if key(x) == best]
    iop = frames[0]['iop']
    r, c = iop[:3] / np.linalg.norm(iop[:3]), iop[3:] / np.linalg.norm(iop[3:])
    n = np.cross(r, c)
    n = n / np.linalg.norm(n)
    pos = np.round(np.array([x['ipp'] @ n for x in frames]), 1)
    if len(np.unique(pos)) < len(pos):
        E = {}
        for x, q in zip(frames, pos):
            E.setdefault(x['echo'], set()).add(q)
        if len(E) > 1:
            e0 = sorted(E, key=lambda e: (-len(E[e]), e))[0]
            keep = [i for i, x in enumerate(frames) if x['echo'] == e0]
            frames, pos = [frames[i] for i in keep], pos[keep]
        seen, keep = set(), []
        for i in sorted(range(len(frames)), key=lambda i: (pos[i], frames[i]['inst'])):
            if pos[i] not in seen:
                seen.add(pos[i])
                keep.append(i)
        info['duplicates_dropped'] = len(frames) - len(keep)
        frames = [frames[i] for i in keep]
    w = np.array([x['ipp'] @ n for x in frames])
    o = np.argsort(w, kind='stable')
    frames, w = [frames[i] for i in o], w[o]
    vol = np.stack([x['px'].astype(np.float32) * np.float32(x['slope']) + np.float32(x['inter']) for x in frames])
    if sum(x['mono1'] for x in frames) * 2 > len(frames):
        vol = vol.max() - vol
    info.update(used=len(frames), tsx='|'.join(sorted(tsx)), shape=list(vol.shape))
    return SimpleNamespace(vol=vol, ipp=np.array([x['ipp'] for x in frames]), r=r, c=c, n=n, w=w,
                           dy=float(frames[0]['ps'][0]), dx=float(frames[0]['ps'][1]),
                           dz=float(np.median(np.diff(w))) if len(w) > 1 else float(frames[0]['thk'] or 3.0),
                           info=info, uid=os.path.basename(os.path.normpath(d)))


def load_mask(path):
    import SimpleITK as sitk
    m = sitk.ReadImage(path)
    return dict(arr=sitk.GetArrayFromImage(m).astype(np.uint8), D=np.array(m.GetDirection(), float).reshape(3, 3),
                o=np.array(m.GetOrigin(), float), sp=np.array(m.GetSpacing(), float))


def mask_at(M, pts):
    q = ((pts - M['o']) @ M['D']) / M['sp']
    idx = np.rint(q).astype(np.int64)
    size = np.array(M['arr'].shape[::-1])
    ok = np.all((idx >= 0) & (idx < size), axis=-1)
    out = np.zeros(pts.shape[:-1], np.uint8)
    ii = idx[ok]
    out[ok] = M['arr'][ii[:, 2], ii[:, 1], ii[:, 0]]
    return out


class PlaneError(ValueError):
    pass


def canon_axes(view, s):
    med, inf, post = np.array([-s, 0.0, 0.0]), np.array([0.0, 0.0, -1.0]), np.array([0.0, 1.0, 0.0])
    return {'sag': (inf, post, med), 'cor': (inf, med, post), 'ax': (post, med, inf)}[view]


def frame_for(V, view, s):
    rt, ct, tt = canon_axes(view, s)
    cos = float(V.n @ tt)
    if abs(cos) < 0.7:
        raise PlaneError(f'series plane is not {view} (normal . expected = {cos:.2f})')
    nc = V.n * np.sign(cos)
    er = rt - (rt @ V.n) * V.n
    er = er / np.linalg.norm(er)
    ec = ct - (ct @ V.n) * V.n
    ec = ec - (ec @ er) * er
    ec = ec / np.linalg.norm(ec)
    return er, ec, nc


def corners(b):
    return np.array([[x, y, z] for x in b[0:2] for y in b[2:4] for z in b[4:6]], float)


def _sample(V, k, C, er, ec, g, sig):
    P = V.ipp[k]
    base = (C - ((C - P) @ V.n) * V.n) - P
    jj = (base @ V.r + g[:, None] * (er @ V.r) + g[None, :] * (ec @ V.r)) / V.dx
    ii = (base @ V.c + g[:, None] * (er @ V.c) + g[None, :] * (ec @ V.c)) / V.dy
    img = V.vol[k]
    H, W = img.shape
    inside = (ii >= -0.5) & (ii <= H - 0.5) & (jj >= -0.5) & (jj <= W - 0.5)
    if not inside.any():
        return np.full(ii.shape, np.nan, np.float32)
    if sig is not None:
        pad = int(4 * max(sig)) + 2
        i0, i1 = max(int(np.floor(ii[inside].min())) - pad, 0), min(int(np.ceil(ii[inside].max())) + pad + 1, H)
        j0, j1 = max(int(np.floor(jj[inside].min())) - pad, 0), min(int(np.ceil(jj[inside].max())) + pad + 1, W)
        sub = ndi.gaussian_filter(img[i0:i1, j0:j1], sig, mode='nearest')
        v = ndi.map_coordinates(sub, [np.clip(ii - i0, 0, i1 - i0 - 1), np.clip(jj - j0, 0, j1 - j0 - 1)], order=1, mode='nearest')
    else:
        v = ndi.map_coordinates(img, [np.clip(ii, 0, H - 1), np.clip(jj, 0, W - 1)], order=1, mode='nearest')
    return np.where(inside, v, np.nan).astype(np.float32)


def crop(V, box, view, s, M=None, out=OUT_PX, margin=MARGIN_MM, max_slices=None, aa=True):
    max_slices = max_slices or MAX_SLICES
    er, ec, nc = frame_for(V, view, s)
    Q = corners(box)
    C = Q.mean(0)
    half = max(np.abs((Q - C) @ er).max(), np.abs((Q - C) @ ec).max()) + margin
    d = 2.0 * half / out
    g = (np.arange(out) - (out - 1) / 2.0) * d
    t = Q @ nc
    t0, t1, tc = t.min() - margin, t.max() + margin, float(C @ nc)
    wc = V.ipp @ nc
    order = np.argsort(wc, kind='stable')
    ws = wc[order]
    cover = float(np.clip((min(t1, ws[-1]) - max(t0, ws[0])) / (t1 - t0), 0, 1))
    n_in = int(((ws >= t0) & (ws <= t1)).sum())
    n_full = max(n_in, min(MIN_SLICES, len(wc))) if V.dz >= THIN_MM else max(1, int(round((t1 - t0) / SLAB_MM)))
    sig = None
    if aa:
        fi, fj = d / V.dy, d / V.dx
        si, sj = (0.5 * np.sqrt(f * f - 1) if f > 1.2 else 0.0 for f in (fi, fj))
        if si > 0 or sj > 0:
            sig = (si, sj)
    if V.dz >= THIN_MM:
        sel = order[(ws >= t0) & (ws <= t1)]
        if len(sel) < MIN_SLICES:
            dist = np.maximum(np.maximum(t0 - wc, wc - t1), 0)
            sel = np.argsort(dist, kind='stable')[:min(MIN_SLICES, len(wc))]
            sel = sel[np.argsort(wc[sel], kind='stable')]
        if len(sel) > max_slices:
            sel = sel[np.round(np.linspace(0, len(sel) - 1, max_slices)).astype(int)]
        groups, centers, mode = [[int(k)] for k in sel], wc[sel], '2d'
    else:
        K = max(1, int(round((t1 - t0) / SLAB_MM)))
        K = min(K, max_slices)
        step = (t1 - t0) / K
        centers, groups, mode = t0 + step * (np.arange(K) + 0.5), [], 'slab'
        for cp in centers:
            mem = np.where(np.abs(wc - cp) <= step / 2)[0]
            if not len(mem):
                k = int(np.argmin(np.abs(wc - cp)))
                mem = np.array([k]) if abs(wc[k] - cp) <= step else np.array([], int)
            groups.append([int(k) for k in mem])
        keep = [i for i, gr in enumerate(groups) if gr]
        if not keep:
            k = int(np.argmin(np.abs(wc - tc)))
            groups, centers, keep = [[k]], np.array([wc[k]]), [0]
        groups, centers = [groups[i] for i in keep], np.asarray(centers)[keep]
    imgs, msks = [], []
    for grp, cp in zip(groups, centers):
        if grp:
            st = np.stack([_sample(V, k, C, er, ec, g, sig) for k in grp])
            with warnings.catch_warnings():
                warnings.simplefilter('ignore', category=RuntimeWarning)
                imgs.append(np.nanmean(st, axis=0) if len(grp) > 1 else st[0])
        else:
            imgs.append(np.full((out, out), np.nan, np.float32))
        if M is not None:
            pts = (C + (cp - tc) * nc) + g[:, None, None] * er + g[None, :, None] * ec
            msks.append(mask_at(M, pts))
    img = np.stack(imgs).astype(np.float32)
    pos = (np.asarray(centers, float) - tc).astype(np.float32)
    meta = dict(d_mm=round(float(d), 4), slice_mm=round(float(np.median(np.diff(centers))) if len(centers) > 1 else float(V.dz), 3),
                mode=mode, n=int(len(centers)), n_in_range=n_in, n_full=int(n_full), cover=round(cover, 3),
                fov=round(float(np.isfinite(img).mean()), 3), aa=None if sig is None else [round(float(x), 2) for x in sig])
    return img, (np.stack(msks) if M is not None else None), pos, meta


def norm_stats(V, box, view, s):
    v = np.array([])
    if box is not None:
        try:
            img = crop(V, box, view, s, None, out=64, margin=0.0, max_slices=16, aa=False)[0]
            v = img[np.isfinite(img)]
        except Exception:
            v = np.array([])
    src = 'bones_box'
    if v.size < 500:
        v, src = V.vol[:, ::4, ::4].ravel(), 'whole_series'
    lo, hi = np.percentile(v, [0.5, 99.5])
    if hi - lo < 1e-3:
        hi = lo + 1.0
    fg = v[v > lo + 0.05 * (hi - lo)]
    p50 = float(np.median(fg)) if fg.size else float(hi)
    return float(lo), float(hi), src, p50


def to_u8(img, lo, hi):
    x = np.clip((img - lo) / (hi - lo), 0.0, 1.0)
    return np.where(np.isfinite(x), x * 255.0 + 0.5, 0).astype(np.uint8)


def enc_stack(u8):
    buf = io.BytesIO()
    im = Image.fromarray(np.ascontiguousarray(u8.reshape(-1, u8.shape[2])), 'L')
    if WEBP:
        im.save(buf, 'WEBP', lossless=True, quality=10, method=1)
    else:
        im.save(buf, 'PNG', compress_level=6)
    return np.frombuffer(buf.getvalue(), np.uint8)


def dec_stack(b, n):
    a = cv2.imdecode(np.frombuffer(np.asarray(b, np.uint8).tobytes(), np.uint8), cv2.IMREAD_GRAYSCALE)
    if a is None:
        raise ValueError('could not decode a crop image')
    return a.reshape(n, -1, a.shape[1])


def decode(z, meta=None):
    keys = list(z.keys()) if hasattr(z, 'keys') else list(z.files)
    if meta is None:
        meta = json.loads(str(z['meta']))
    A = {}
    for name, cm in meta['crops'].items():
        n = int(cm['n'])
        if f'{name}__imgw' in keys:
            A[f'{name}__img'] = dec_stack(z[f'{name}__imgw'], n)
        elif f'{name}__img' in keys:
            A[f'{name}__img'] = np.asarray(z[f'{name}__img'])
        if f'{name}__mskz' in keys:
            A[f'{name}__msk'] = np.frombuffer(zlib.decompress(np.asarray(z[f'{name}__mskz'], np.uint8).tobytes()),
                                              np.uint8).reshape(n, OUT_PX, OUT_PX)
        elif f'{name}__msk' in keys:
            A[f'{name}__msk'] = np.asarray(z[f'{name}__msk'])
        if f'{name}__pos' in keys:
            A[f'{name}__pos'] = np.asarray(z[f'{name}__pos'])
    return A, meta


def read_study(path):
    with np.load(path) as z:
        return decode(z)


def _centroid(m, lab, min_px=150):
    z = np.argwhere(m == lab)
    return z.mean(0) if len(z) >= min_px else None


def orient_checks(msk, view, roi):
    c = {b: _centroid(msk, lab) for lab, b in ((1, 'fem'), (2, 'pat'), (3, 'fib'), (4, 'tib'))}
    out = {}
    if view in ('sag', 'cor') and c['fem'] is not None and c['tib'] is not None:
        out['femur_above_tibia'] = bool(c['fem'][1] < c['tib'][1])
    if c['pat'] is not None and c['fem'] is not None:
        if view == 'sag':
            out['patella_anterior'] = bool(c['pat'][2] < c['fem'][2])
        if view == 'ax':
            out['patella_anterior'] = bool(c['pat'][1] < c['fem'][1])
    if c['fib'] is not None and c['tib'] is not None:
        if view in ('cor', 'ax'):
            out['fibula_lateral'] = bool(c['fib'][2] < c['tib'][2])
        if view == 'sag':
            out['fibula_lateral'] = bool(c['fib'][0] < c['tib'][0])
    if roi == 'medial_TF':
        out['no_fibula_medial'] = bool((msk == 3).sum() < 150)
    return out


def align_shift(u8, msk, d, valid=None, max_mm=8.0):
    b = msk > 0
    ok = [k for k in range(len(b)) if b[k].sum() >= 200]
    if len(ok) < 2:
        return None
    R = max(1, int(round(max_mm / d)))
    H, W = u8.shape[1:]
    N = (H + 2 * R, W + 2 * R)
    frame = np.zeros((H, W), bool)
    frame[2:-2, 2:-2] = True
    acc, nb = 0, 0
    for k in ok:
        im = ndi.gaussian_filter(u8[k].astype(np.float32), 1.0)
        G = np.hypot(ndi.sobel(im, 0), ndi.sobel(im, 1))
        if valid is not None:
            G[~ndi.binary_erosion(valid[k], iterations=3)] = 0
        B = (b[k] & ~ndi.binary_erosion(b[k]) & frame).astype(np.float32)
        acc = acc + np.fft.rfft2(G, N) * np.conj(np.fft.rfft2(B, N))
        nb += B.sum()
    cc = np.fft.irfft2(acc, N) / max(nb, 1.0)
    sc = cc[np.ix_(np.arange(-R, R + 1) % N[0], np.arange(-R, R + 1) % N[1])]
    i, j = np.unravel_index(int(np.argmax(sc)), sc.shape)
    s0 = sc[R, R]
    return dict(dr_mm=round(float((i - R) * d), 2), dc_mm=round(float((j - R) * d), 2),
                gain=round(float(sc[i, j] / s0), 4) if s0 > 0 else None)


def _seq_info(series_rows, uid):
    try:
        r = series_rows[series_rows.SeriesInstanceUID.astype(str) == uid].iloc[0]
    except Exception:
        return {}

    def g(k):
        return r[k] if k in r.index else None
    tr, te = _f(g('TR')), _f(g('TE'))
    txt = f"{g('image_type') or ''} {g('series_desc') or ''}".upper()
    seq = str(g('scanning_sequence') or '').upper()
    return dict(desc=str(g('series_desc'))[:40], tr=tr, te=te,
                gre=bool(('GR' in seq and 'SE' not in seq) or any(w in txt for w in GRE_WORDS)),
                kind=None if tr is None or tr != tr else ('T1' if tr < 1000 else 'PD' if (te is not None and te <= 60) else 'T2'))


def study_crops(sid, series_rows, boxes, lat, series_dir, mask_path=None, ref_uid=None, box_src=None, crops=CROPS):
    s = 1.0 if lat == 'L' else -1.0
    M = load_mask(mask_path) if mask_path and os.path.exists(mask_path) else None
    ranked = rank_series(series_rows)
    A, used = {}, set()
    meta = dict(StudyInstanceUID=sid, version=VERSION, lat=lat, side_sign=s, mask=M is not None, slots={}, crops={}, errors=[])
    for slot in SLOTS:
        need = [r for r, sl in crops if sl == slot and boxes.get(r) is not None]
        if not need:
            continue
        labelled = [u for u in ranked.get(slot, [])[:3] if u not in used]
        others = [u for sl2, lst in ranked.items()
                  if sl2[-3:] == slot[-3:] and VIEW[sl2] != VIEW[slot]
                  for u in lst if u not in used and u not in labelled][:2]
        cands = labelled + others
        if not cands:
            meta['slots'][slot] = dict(missing='no series of this kind')
            continue
        V, tried = None, []
        for uid in cands:
            try:
                V = load_series(os.path.join(series_dir, uid))
                frame_for(V, VIEW[slot], s)
                break
            except Exception as e:
                tried.append(f'{uid[-10:]}: {type(e).__name__}: {str(e)[:100]}')
                V = None
        if V is None:
            if labelled:
                meta['slots'][slot] = dict(missing='no usable series', tried=tried)
                meta['errors'].append(f'{slot}: no usable series ({"; ".join(tried)})')
            else:
                meta['slots'][slot] = dict(missing='no series of this kind')
            continue
        used.add(V.uid)
        lo, hi, nsrc, p50 = norm_stats(V, boxes.get('bones'), VIEW[slot], s)
        sm = dict(uid=V.uid, rank=cands.index(V.uid), relabelled=V.uid not in ranked.get(slot, []), lo=round(lo, 3),
                  hi=round(hi, 3), p50=round(p50, 3), norm=nsrc, dz=round(V.dz, 3), dy=round(V.dy, 4), dx=round(V.dx, 4),
                  is_ref=bool(ref_uid is not None and V.uid == ref_uid), info=V.info, failed_before=tried,
                  seq=_seq_info(series_rows, V.uid))
        for roi in need:
            name = f'{roi}__{slot}'
            try:
                img, msk, pos, cm = crop(V, boxes[roi], VIEW[slot], s, M, max_slices=MAX_LARGE if roi in LARGE_ROIS else MAX_SLICES)
                u8 = to_u8(img, lo, hi)
                A[f'{name}__imgw'], A[f'{name}__pos'] = enc_stack(u8), pos.astype(np.float16)
                if msk is not None:
                    A[f'{name}__mskz'] = np.frombuffer(zlib.compress(np.ascontiguousarray(msk).tobytes(), 6), np.uint8)
                    cm['checks'] = orient_checks(msk, VIEW[slot], roi)
                    cm['bone_px'] = {b: int((msk == lab).sum()) for lab, b in ((1, 'femur'), (2, 'patella'), (3, 'fibula'), (4, 'tibia'))}
                    if roi == ANCHOR[slot]:
                        sm['align'] = align_shift(u8, msk, cm['d_mm'], np.isfinite(img))
                cm.update(series=V.uid, slot=slot, roi=roi, box_source=(box_src or {}).get(roi))
                meta['crops'][name] = cm
            except Exception as e:
                meta['errors'].append(f'{name}: {type(e).__name__}: {str(e)[:150]}')
        meta['slots'][slot] = sm
        del V
    return A, meta


def js(o):
    return o.item() if hasattr(o, 'item') else str(o)

In [ ]:
%%writefile knee/data.py
import math
import numpy as np
import torch
import torch.nn.functional as F
from . import crops


def slice_index(S, K, train, rng):
    if S <= K:
        return np.arange(S)
    if not train:
        return np.unique(np.round(np.linspace(0, S - 1, K)).astype(int))
    edges = np.linspace(0, S, K + 1)
    return np.array([rng.integers(int(math.floor(a)), max(int(math.floor(a)) + 1, int(math.ceil(b))))
                     for a, b in zip(edges[:-1], edges[1:])]).clip(0, S - 1)


class StudyData:

    def __init__(self, crops_dir, crop_names, K_train=8, K_eval=12, min_fov=0.25):
        self.dir, self.names = crops_dir, list(crop_names)
        self.K_train, self.K_eval, self.min_fov = K_train, K_eval, min_fov

    def load(self, sid, train, rng):
        A, meta = crops.read_study(f'{self.dir}/{sid}.npz')
        K = self.K_train if train else self.K_eval
        H = crops.OUT_PX
        x = np.zeros((len(self.names), K, 3, H, H), np.uint8)
        valid = np.zeros((len(self.names), K), bool)
        pos = np.zeros((len(self.names), K), np.float32)
        fov = np.zeros(len(self.names), np.float32)
        for j, n in enumerate(self.names):
            if f'{n}__img' not in A:
                continue
            fov[j] = float(meta['crops'].get(n, {}).get('fov', 1.0))
            if fov[j] < self.min_fov:
                continue
            img, p = A[f'{n}__img'], A[f'{n}__pos'].astype(np.float32)
            S = len(img)
            idx = slice_index(S, K, train, rng)
            for t, i in enumerate(idx):
                x[j, t, 0], x[j, t, 1], x[j, t, 2] = img[max(i - 1, 0)], img[i], img[min(i + 1, S - 1)]
            valid[j, :len(idx)] = True
            pos[j, :len(idx)] = p[idx] / 50.0
        return x, valid, pos, fov


def augment_gpu(x, valid, gen=None, rot_deg=10.0, scale=0.1, shift=0.06, contrast=0.15, bright=0.08, gamma=0.15, drop_crop=0.05):
    B, Cn, K = valid.shape
    dev = x.device

    def r(*s):
        return torch.rand(*s, device=dev, generator=gen)
    ang = (r(B, Cn) * 2 - 1) * math.radians(rot_deg)
    sc = 1 + (r(B, Cn) * 2 - 1) * scale
    tx, ty = (r(B, Cn) * 2 - 1) * shift * 2, (r(B, Cn) * 2 - 1) * shift * 2
    th = torch.stack([torch.stack([torch.cos(ang) / sc, -torch.sin(ang) / sc, tx], -1),
                      torch.stack([torch.sin(ang) / sc, torch.cos(ang) / sc, ty], -1)], -2)
    m = valid.reshape(-1)
    flat = x.reshape(B * Cn * K, *x.shape[3:])[m]
    thf = th[:, :, None].expand(B, Cn, K, 2, 3).reshape(-1, 2, 3)[m]
    grid = F.affine_grid(thf, flat.shape, align_corners=False)
    flat = F.grid_sample(flat, grid, mode='bilinear', padding_mode='zeros', align_corners=False)
    c = (1 + (r(B, Cn) * 2 - 1) * contrast)[:, :, None].expand(B, Cn, K).reshape(-1)[m]
    b = ((r(B, Cn) * 2 - 1) * bright)[:, :, None].expand(B, Cn, K).reshape(-1)[m]
    g = (1 + (r(B, Cn) * 2 - 1) * gamma)[:, :, None].expand(B, Cn, K).reshape(-1)[m]
    flat = (flat.clamp(0, 1) ** g[:, None, None, None] * c[:, None, None, None] + b[:, None, None, None]).clamp(0, 1)
    out = torch.zeros_like(x.reshape(B * Cn * K, *x.shape[3:]))
    out[m] = flat
    keep = r(B, Cn) >= drop_crop
    has = (valid.any(-1) & keep).any(-1, keepdim=True)
    keep = torch.where(has, keep, torch.ones_like(keep))
    return out.reshape(x.shape), valid & keep[:, :, None]

In [ ]:
%%writefile knee/model.py
import math
import numpy as np
import torch
import torch.nn as nn
import timm

PRIOR_BIAS = {0: 0.0, 1: -1.0, 2: -3.0}


class KneeNet(nn.Module):

    def __init__(self, backbone, n_crops, relevance, d=256, n_layers=1, heads=4, drop=0.1):
        super().__init__()
        self.enc = timm.create_model(backbone, pretrained=False, num_classes=0, in_chans=3)
        nf = self.enc.num_features
        self.proj = nn.Sequential(nn.Linear(nf, d), nn.LayerNorm(d))
        self.crop_emb = nn.Embedding(n_crops, d)
        self.pos = nn.Sequential(nn.Linear(1, d // 4), nn.GELU(), nn.Linear(d // 4, d))
        layer = nn.TransformerEncoderLayer(d, heads, 2 * d, dropout=drop, batch_first=True, norm_first=True)
        self.mix = nn.TransformerEncoder(layer, n_layers, enable_nested_tensor=False) if n_layers else None
        R = np.asarray(relevance)
        L = R.shape[0]
        self.q = nn.Parameter(torch.randn(L, d) * 0.02)
        self.bias = nn.Parameter(torch.tensor(np.vectorize(PRIOR_BIAS.get)(R), dtype=torch.float32))
        self.drop = nn.Dropout(0.2)
        self.head_w = nn.Parameter(torch.randn(L, d) * 0.02)
        self.head_b = nn.Parameter(torch.zeros(L))
        self.register_buffer('mean', torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))
        self.register_buffer('std', torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))
        self.d = d

    def forward(self, x, valid, pos):
        B, Cn, K = valid.shape
        flat = x.reshape(B * Cn * K, *x.shape[3:])[valid.reshape(-1)]
        f = self.proj(self.enc((flat - self.mean) / self.std).float())
        tok = torch.zeros(B * Cn * K, self.d, device=x.device, dtype=f.dtype)
        tok[valid.reshape(-1)] = f
        tok = tok.view(B, Cn, K, self.d) + self.crop_emb.weight[None, :, None, :] + self.pos(pos[..., None].float())
        tok = tok.view(B, Cn * K, self.d)
        m = valid.view(B, Cn * K)
        if self.mix is not None:
            tok = self.mix(tok, src_key_padding_mask=~m)
        crop_of = torch.arange(Cn, device=x.device).repeat_interleave(K)
        att = torch.einsum('bnd,ld->bln', tok, self.q) / math.sqrt(self.d) + self.bias[:, crop_of][None]
        att = att.masked_fill(~m[:, None, :], -10000.0).softmax(-1)
        pooled = self.drop(torch.einsum('bln,bnd->bld', att, tok))
        return (pooled * self.head_w).sum(-1) + self.head_b, att


def build(cfg, relevance):
    return KneeNet(cfg['backbone'], cfg['n_crops'], relevance, d=cfg.get('d', 256), n_layers=cfg.get('n_layers', 1),
                   heads=cfg.get('heads', 4))

In [ ]:
%%writefile knee/fit.py
import os
import json
import time
import math
import argparse
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import roc_auc_score
from timm.utils import ModelEmaV3
from knee.findings import LABELS, relevance
from knee.data import StudyData, augment_gpu
from knee.model import build

ap = argparse.ArgumentParser()
ap.add_argument('--run', required=True)
ap.add_argument('--index', type=int, required=True)
ap.add_argument('--name', required=True)
ap.add_argument('--deadline', type=float, default=0)
a = ap.parse_args()
cfg = json.load(open(f'{a.run}/config.json'))
torch.manual_seed(cfg['seed'] + a.index)
np.random.seed(cfg['seed'] + a.index)
dev = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
AMP = dev.type == 'cuda'
OUT = f'{a.run}/{a.name}'
os.makedirs(OUT, exist_ok=True)


def log(*s):
    print(f'[{a.name}]', *s, flush=True)


T = pd.read_csv(f'{a.run}/table.csv')
tr, va = T[T.fold != a.index].reset_index(drop=True), T[T.fold == a.index].reset_index(drop=True)
SD = StudyData(cfg['crops_dir'], cfg['crop_names'], cfg['K_train'], cfg['K_eval'], cfg['min_fov'])


class Studies(Dataset):

    def __init__(self, tab, train):
        self.t, self.train, self.rng = tab, train, None

    def __len__(self):
        return len(self.t)

    def __getitem__(self, i):
        if self.rng is None:
            self.rng = np.random.default_rng((cfg['seed'] * 1000 + a.index * 100 + os.getpid()) % 2 ** 32)
        r = self.t.iloc[i]
        x, valid, pos, fov = SD.load(r.StudyInstanceUID, self.train, self.rng)
        y = r[LABELS].values.astype(np.float32)
        w = np.full(len(LABELS), cfg['gold_w'] if r.is_gold == 1 else 1.0, np.float32)
        return torch.from_numpy(x), torch.from_numpy(valid), torch.from_numpy(pos), torch.from_numpy(y), torch.from_numpy(w), i


def loader(tab, train, bs):
    return DataLoader(Studies(tab, train), batch_size=bs, shuffle=train, drop_last=train, num_workers=cfg['workers'],
                      pin_memory=AMP, persistent_workers=cfg['workers'] > 0, prefetch_factor=2 if cfg['workers'] > 0 else None)


def autocast():
    return torch.autocast(device_type='cuda', dtype=torch.float16) if AMP else torch.autocast(device_type='cpu', enabled=False)


model = build(cfg, relevance(cfg['crop_names'])).to(dev)
sd = torch.load(cfg['pretrained'], map_location='cpu')
res = model.enc.load_state_dict(sd, strict=False)
n_enc = len([k for k in model.enc.state_dict() if not k.endswith('num_batches_tracked')])
missing = [k for k in res.missing_keys if not k.endswith('num_batches_tracked')]
if len(missing) > 0.05 * n_enc:
    raise RuntimeError(f'{cfg["pretrained"]} does not fit the {cfg["backbone"]} encoder (missing {missing[:5]})')
enc_p = list(model.enc.parameters())
enc_ids = {id(q) for q in enc_p}
opt = torch.optim.AdamW([{'params': enc_p, 'lr': cfg['lr_enc']},
                         {'params': [q for q in model.parameters() if id(q) not in enc_ids], 'lr': cfg['lr_head']}],
                        weight_decay=cfg['wd'])
base_lr = [g['lr'] for g in opt.param_groups]
scaler = torch.amp.GradScaler('cuda', enabled=AMP)
ema = ModelEmaV3(model, decay=cfg['ema']) if cfg.get('ema') else None
dl_tr = loader(tr, True, cfg['batch'])
steps_per_epoch = len(dl_tr) // cfg['accum']
total = max(1, steps_per_epoch * cfg['epochs'])
warm = min(300, total // 10)


def set_lr(step):
    f = step / max(1, warm) if step < warm else 0.02 + 0.98 * 0.5 * (1 + math.cos(math.pi * (step - warm) / max(1, total - warm)))
    for g, b in zip(opt.param_groups, base_lr):
        g['lr'] = b * f


def to_dev(batch):
    x, valid, pos, y, w, idx = batch
    return (x.to(dev, non_blocking=True).float().div_(255), valid.to(dev, non_blocking=True), pos.to(dev, non_blocking=True),
            y.to(dev, non_blocking=True), w.to(dev, non_blocking=True), idx)


@torch.no_grad()
def predict(net, tab):
    net.eval()
    P = np.zeros((len(tab), len(LABELS)), np.float32)
    for batch in loader(tab, False, cfg['batch_eval']):
        x, valid, pos, y, w, idx = to_dev(batch)
        with autocast():
            lg, _ = net(x, valid, pos)
        P[idx.numpy()] = torch.sigmoid(lg.float()).cpu().numpy()
    return P


def auc(y, p):
    return roc_auc_score(y, p) if 0 < y.sum() < len(y) else np.nan


def metrics(tab, P):
    Y = tab[LABELS].values.astype(np.float32)
    Pc = np.clip(P, 1e-06, 1 - 1e-06)
    W = np.where(tab.is_gold.values[:, None] == 1, cfg['gold_w'], 1.0) * np.ones_like(Y)
    bce = float(-(W * (Y * np.log(Pc) + (1 - Y) * np.log(1 - Pc))).sum() / W.sum())
    return bce, float(np.nanmean([auc(Y[:, j] >= 0.5, P[:, j]) for j in range(len(LABELS))]))


log(f'{len(tr)} studies to train, {len(va)} to validate | {cfg["epochs"]} epochs x {steps_per_epoch} steps | {dev}')
best, step, t_start, hist = None, 0, time.time(), []
for ep in range(cfg['epochs']):
    model.train()
    t_ep, seen, run_loss = time.time(), 0, 0.0
    for i, batch in enumerate(dl_tr):
        x, valid, pos, y, w, _ = to_dev(batch)
        x, valid = augment_gpu(x, valid)
        with autocast():
            lg, _ = model(x, valid, pos)
            loss = (F.binary_cross_entropy_with_logits(lg.float(), y, reduction='none') * w).sum() / w.sum()
        scaler.scale(loss / cfg['accum']).backward()
        run_loss += loss.item()
        seen += len(y)
        if (i + 1) % cfg['accum'] == 0:
            set_lr(step)
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt)
            scaler.update()
            opt.zero_grad(set_to_none=True)
            step += 1
            if ema is not None:
                ema.update(model, step=step)
        if (i + 1) % cfg['log_every'] == 0:
            el = time.time() - t_ep
            log(f'epoch {ep + 1}/{cfg["epochs"]} step {i + 1}/{len(dl_tr)} | loss {run_loss / (i + 1):.4f} | '
                f'{seen / el:.1f} studies/s | {(len(dl_tr) - i - 1) * el / (i + 1) / 60:.1f} min left in epoch')
    net = ema.module if ema is not None else model
    rec = dict(epoch=ep + 1, train_loss=run_loss / max(1, len(dl_tr)))
    if len(va):
        P = predict(net, va)
        rec['val_loss'], rec['val_auc'] = metrics(va, P)
        g = va.is_gold.values == 1
        gauc = [auc(va.loc[g, l].values >= 0.5, P[g, j]) for j, l in enumerate(LABELS)] if g.sum() >= 4 else []
        gauc = [v for v in gauc if v == v]
        rec['gold_auc'] = float(np.mean(gauc)) if gauc else None
    rec['min'] = (time.time() - t_ep) / 60
    hist.append(rec)
    saved = not len(va) or best is None or rec['val_loss'] < best['val_loss']
    if saved:
        best = rec
        torch.save(dict(state_dict=net.state_dict(), epoch=ep + 1), f'{OUT}/best.pt')
    log(f'epoch {ep + 1} | train loss {rec["train_loss"]:.4f}'
        + (f' | val loss {rec["val_loss"]:.4f} | val auc {rec["val_auc"]:.4f}' if len(va) else '')
        + (f' | gold auc {rec["gold_auc"]:.3f}' if rec.get('gold_auc') is not None else '')
        + (' | saved' if saved else '') + f' | {rec["min"]:.1f} min')
    json.dump(hist, open(f'{OUT}/history.json', 'w'), indent=1)
    if a.deadline and time.time() + (time.time() - t_ep) * 1.2 > a.deadline and ep + 1 < cfg['epochs']:
        log('stopping: the next epoch would not end before the time limit')
        break
if len(va):
    model.load_state_dict(torch.load(f'{OUT}/best.pt', map_location='cpu')['state_dict'])
    P = predict(model, va)
    oof = va[['StudyInstanceUID']].copy()
    for j, l in enumerate(LABELS):
        oof[l] = P[:, j]
    oof['fold'] = a.index
    oof.to_csv(f'{OUT}/oof.csv', index=False)
json.dump(dict(best=best, epochs_run=len(hist), minutes=(time.time() - t_start) / 60), open(f'{OUT}/done.json', 'w'), indent=1)
log(f'done | best epoch {best["epoch"]} | {(time.time() - t_start) / 60:.0f} min')

In [ ]:
%%writefile knee/launch.py
import os
import sys
import time
import subprocess

ROOT = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))


def worker(module, *args):
    return [sys.executable, '-m', f'knee.{module}', *[str(x) for x in args]]


def env(**extra):
    e = dict(os.environ, PYTHONPATH=ROOT + os.pathsep + os.environ.get('PYTHONPATH', ''))
    e.update({k: str(v) for k, v in extra.items()})
    return e


def _start(cmd, e, log):
    return dict(proc=subprocess.Popen(cmd, stdout=open(log, 'w'), stderr=subprocess.STDOUT, env=e, cwd=ROOT),
                log=log, pos=0, rest='')


def _pump(p):
    with open(p['log'], errors='replace') as fh:
        fh.seek(p['pos'])
        text = p['rest'] + fh.read()
        p['pos'] = fh.tell()
    lines = text.split('\n')
    p['rest'] = lines[-1]
    for line in lines[:-1]:
        if line.startswith('['):
            print(line, flush=True)
    if p['proc'].poll() is None:
        return None
    if p['rest'].startswith('['):
        print(p['rest'], flush=True)
    return p['proc'].returncode


def _stop(running, p):
    for q in running:
        q['proc'].kill()
    tail = open(p['log'], errors='replace').read()[-4000:]
    raise RuntimeError(f'worker stopped with exit code {p["proc"].returncode} ({p["log"]}):\n{tail}')


def run_parallel(jobs, strict=True, poll=15):
    running = [_start(cmd, e, log) for cmd, e, log in jobs]
    failed = []
    while running:
        time.sleep(poll)
        for p in list(running):
            code = _pump(p)
            if code is None:
                continue
            running.remove(p)
            if code != 0:
                if strict:
                    _stop(running, p)
                failed.append(p)
    for p in failed:
        tail = open(p['log'], errors='replace').read()[-600:].replace('\n', ' | ')
        print(f'[worker] exit code {p["proc"].returncode}: {tail}', flush=True)
    return len(failed)


def run_queue(jobs, n_slots, deadline=0, poll=15):
    queue, running = list(jobs), {}
    while queue or running:
        if queue and deadline and time.time() > deadline - 20 * 60:
            print(f'[queue] not started, too close to the time limit: {[os.path.dirname(log) for _, _, log in queue]}', flush=True)
            queue = []
        for slot in range(n_slots):
            if slot not in running and queue:
                cmd, e, log = queue.pop(0)
                running[slot] = _start(cmd, dict(e, CUDA_VISIBLE_DEVICES=str(slot)), log)
        time.sleep(poll)
        for slot, p in list(running.items()):
            code = _pump(p)
            if code is None:
                continue
            del running[slot]
            if code != 0:
                _stop(list(running.values()), p)

In [ ]:
%%writefile configs/kneenet.json
{
 "backbone": "resnet18d",
 "d": 256,
 "n_layers": 1,
 "heads": 4,
 "epochs": 12,
 "batch": 4,
 "accum": 2,
 "batch_eval": 8,
 "K_train": 8,
 "K_eval": 12,
 "min_fov": 0.25,
 "lr_enc": 0.0002,
 "lr_head": 0.0005,
 "wd": 0.02,
 "ema": 0.999,
 "gold_w": 3.0,
 "workers": 2,
 "log_every": 250,
 "seed": 42
}

In [ ]:
%%writefile train.py
import os
import json
import time
import argparse
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import roc_auc_score
from knee import crops
from knee.findings import LABELS, multilabel_folds
from knee.launch import worker, env, run_queue


def study_table(labels_csv, crops_dir, seed):
    L = pd.read_csv(labels_csv)
    missing = [c for c in ['StudyInstanceUID', 'is_gold'] + LABELS if c not in L]
    if missing:
        raise RuntimeError(f'{labels_csv} has no column {missing}')
    T = L[[os.path.exists(f'{crops_dir}/{s}.npz') for s in L.StudyInstanceUID]].reset_index(drop=True)
    if len(T) < 0.97 * len(L):
        raise RuntimeError(f'only {len(T)} of {len(L)} labelled studies have a crops file in {crops_dir}')
    if 'fold' in T and T.fold.notna().all():
        T['fold'] = T.fold.astype(int)
    else:
        T['fold'] = multilabel_folds(np.c_[T[LABELS].values >= 0.5, T.is_gold.values == 1], k=5, seed=seed)
    return T


def auc(y, p):
    return roc_auc_score(y, p) if 0 < y.sum() < len(y) else np.nan


def out_of_fold(out, T, runs):
    O = pd.concat([pd.read_csv(f'{out}/{name}/oof.csv') for name, _ in runs])
    O.to_csv(f'{out}/oof.csv', index=False)
    E = T.merge(O, on='StudyInstanceUID', suffixes=('_y', ''))
    g = (E.is_gold == 1).values
    rows = {'gold studies, radiologist labels': {l: auc(E.loc[g, f'{l}_y'].values >= 0.5, E.loc[g, l].values) for l in LABELS},
            'other studies, training labels': {l: auc(E.loc[~g, f'{l}_y'].values >= 0.5, E.loc[~g, l].values) for l in LABELS}}
    if all(f'report_{l}' in E for l in LABELS):
        rows['other studies, report labels'] = {l: auc(E.loc[~g, f'report_{l}'].values >= 0.5, E.loc[~g, l].values) for l in LABELS}
    tab = pd.DataFrame(rows).T
    tab.insert(0, 'macro', tab[LABELS].mean(axis=1))
    tab.to_csv(f'{out}/auc.csv')
    print(f'out-of-fold AUC, {len(E)} studies ({int(g.sum())} gold)')
    print(tab.round(3).to_string())


def main(argv=None):
    ap = argparse.ArgumentParser()
    ap.add_argument('--config', default=os.path.join(os.path.dirname(os.path.abspath(__file__)), 'configs', 'kneenet.json'))
    ap.add_argument('--labels', required=True)
    ap.add_argument('--crops', required=True)
    ap.add_argument('--pretrained', required=True)
    ap.add_argument('--out', required=True)
    ap.add_argument('--folds', type=int, nargs='*', default=[0, 1, 2, 3, 4])
    ap.add_argument('--full', type=int, default=0)
    ap.add_argument('--gpus', type=int, default=0)
    ap.add_argument('--hours', type=float, default=0)
    a = ap.parse_args(argv)
    t0 = time.time()
    n_slots = a.gpus or max(1, torch.cuda.device_count())
    cfg = json.load(open(a.config))
    cfg.update(crop_names=list(crops.CROP_NAMES), n_crops=len(crops.CROP_NAMES), crops_dir=os.path.abspath(a.crops),
               pretrained=os.path.abspath(a.pretrained))
    os.makedirs(a.out, exist_ok=True)
    T = study_table(a.labels, a.crops, cfg['seed'])
    T.to_csv(f'{a.out}/table.csv', index=False)
    json.dump(cfg, open(f'{a.out}/config.json', 'w'), indent=1)
    runs = [(f'full{k}', -k) for k in range(1, a.full + 1)] if a.full else [(f'fold{k}', k) for k in a.folds]
    deadline = t0 + a.hours * 3600 if a.hours else 0
    todo = [(n, i) for n, i in runs if not os.path.exists(f'{a.out}/{n}/done.json')]
    print(f'{len(T)} studies ({int(T.is_gold.sum())} gold) | runs {[n for n, _ in todo]} | {n_slots} at a time', flush=True)
    for name, _ in todo:
        os.makedirs(f'{a.out}/{name}', exist_ok=True)
    run_queue([(worker('fit', '--run', os.path.abspath(a.out), '--index', i, '--name', n, '--deadline', deadline), env(),
                f'{a.out}/{n}/train.log') for n, i in todo], n_slots, deadline)
    if not a.full and all(os.path.exists(f'{a.out}/{n}/oof.csv') for n, _ in runs):
        out_of_fold(a.out, T, runs)
    print(f'training finished in {(time.time() - t0) / 3600:.1f} h', flush=True)


if __name__ == '__main__':
    main()

In [ ]:
sys.path.insert(0, CODE)
import train
args = ['--labels', labels, '--crops', crops, '--pretrained', pretrained, '--out', f'/kaggle/working/{RUN}', '--hours', '11.3']
train.main(args + (['--full', '2'] if RUN == 'consensus_full' else []))